In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report, RocCurveDisplay)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

import optuna

In [32]:
df = pd.read_csv('ENOE_SDEMT226.csv', encoding='latin-1')

C:\Users\1\AppData\Local\Temp\ipykernel_10876\4252273001.py:1: DtypeWarning: Columns (22,23,24,25,26,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('ENOE_SDEMT226.csv', encoding='latin-1')


## Variables y limpieza

Aqui se ponene el nombre de columnas sin espacios en las orillas y en minusculas, se seleccionan las columnas a usar, y volvemos numericos todas las columnas teniendo en cuenta que todas las columnas seleccionadas ya son numericas.

In [33]:
# Normalizar los nombres de las columnas
df.columns = df.columns.str.strip().str.lower()

# Seleccionar las columnas usadas en el proyecto
columnas = [
    'r_def', 'clase2', 'emp_ppal',
    'cve_ent', 'cve_mun', 'fac_tri',
    'eda', 'anios_esc', 'hrsocup', 'ingocup',
    'sex', 'e_con', 'niv_ins', 'cs_p12', 'cs_p17',
    't_loc_tri', 'ur', 'zona', 'rama', 'scian', 'dur9c'
]

# Quitamos ing7c, pos_ocu

df = df[columnas].copy()

# Convertir las columnas a números
# Los valores vacíos o que no se puedan convertir quedan como NaN
for columna in columnas:
    df[columna] = pd.to_numeric(df[columna], errors='coerce')

print('Filas originales:', len(df))

Filas originales: 417128


FIltrado  de columnas

In [34]:
# Entrevistas completas
df = df[df['r_def'] == 0]
print('Entrevistas completas:', len(df))

# Personas ocupadas
df = df[df['clase2'] == 1]
print('Personas ocupadas:', len(df))

# Personas de 15 años o más, sin el código 99
df = df[(df['eda'] >= 15) & (df['eda'] != 99)]
print('Edad válida:', len(df))

# Solo empleo informal o formal
df = df[df['emp_ppal'].isin([1, 2])].copy()
print('Condición de empleo válida:', len(df))

Entrevistas completas: 416393
Personas ocupadas: 190325
Edad válida: 189353
Condición de empleo válida: 189353


Limpieza de valores y creacion de nuevas variables

In [35]:
# Cambiar los códigos de no especificado por valores nulos
df['anios_esc'] = df['anios_esc'].replace(99, np.nan)
df['e_con'] = df['e_con'].replace(9, np.nan)
df['cs_p12'] = df['cs_p12'].replace(9, np.nan)
df['cs_p17'] = df['cs_p17'].replace(9, np.nan)

# Crear variables derivadas
df['log_ing'] = np.log(df['ingocup'] + 1)
df['ing_cero'] = (df['ingocup'] == 0).astype(int)
df['hrs_cero'] = (df['hrsocup'] == 0).astype(int)

# Columnas que P02 revisa antes de eliminar nulos
predictoras_limpieza = [
    'eda', 'anios_esc', 'hrsocup', 'log_ing',
    'ing_cero', 'hrs_cero',
    'sex', 'e_con', 'niv_ins', 'cs_p12', 'cs_p17',
    't_loc_tri', 'ur', 'zona', 'rama', 'scian', 'dur9c'
]

print('Nulos antes de eliminar filas:')
print(df[predictoras_limpieza].isnull().sum())

antes = len(df)
df = df.dropna(subset=predictoras_limpieza).copy()

print('Filas eliminadas por nulos:', antes - len(df))
print('Filas después de limpieza:', len(df))

Nulos antes de eliminar filas:
eda            0
anios_esc    622
hrsocup        0
log_ing        0
ing_cero       0
hrs_cero       0
sex            0
e_con         28
niv_ins        0
cs_p12        18
cs_p17        22
t_loc_tri      0
ur             0
zona           0
rama           0
scian          0
dur9c          0
dtype: int64
Filas eliminadas por nulos: 680
Filas después de limpieza: 188673


In [36]:
# Crear la variable objetivo
# 1 = informal, 0 = formal
df['y'] = (df['emp_ppal'] == 1).astype(int)

# Ya se usaron para filtrar o construir y
df = df.drop(columns=['r_def', 'clase2', 'emp_ppal'])

# Claves geográficas
df['cve_ent'] = df['cve_ent'].astype(int)
df['cve_mun'] = df['cve_mun'].fillna(0).astype(int)

# Convertir los códigos categóricos a texto
categoricas = [
    'sex', 'e_con', 'niv_ins', 'cs_p12', 'cs_p17',
    't_loc_tri', 'ur', 'zona', 'rama', 'scian', 'dur9c'
]

for columna in categoricas:
    df[columna] = df[columna].astype(int).astype(str)

# Se usa en la comparación de variables, no en el modelo final
df['eda2'] = df['eda'] ** 2

df = df.reset_index(drop=True)

In [40]:
# Crear los tres conjuntos
df_nacional = df.copy()

df_jalisco = df[df['cve_ent'] == 14].copy()

municipios_amg5 = [39, 120, 98, 101, 97]
df_amg5 = df_jalisco[df_jalisco['cve_mun'].isin(municipios_amg5)].copy()

# Crear la variable geográfica para cada conjunto
df_amg5['geo'] = df_amg5['cve_mun'].astype(str)
df_jalisco['geo'] = df_jalisco['cve_mun'].astype(str)
df_nacional['geo'] = df_nacional['cve_ent'].astype(str)

print('AMG5:', len(df_amg5), 'filas')
print('Jalisco:', len(df_jalisco), 'filas')
print('Nacional:', len(df_nacional), 'filas')

AMG5: 3959 filas
Jalisco: 5882 filas
Nacional: 188673 filas


## EDA